# 04 — Cierre y conclusiones defendibles

Este notebook lee el cierre y los reportes existentes. No entrena, no ejecuta búsquedas y no vuelve a consultar etiquetas para seleccionar modelos. La comparación principal es utilidad multiobjetivo histórica por regresor; precisión, estabilidad y costo la complementan. La búsqueda A/B es secundaria.

In [ ]:
%matplotlib inline
%config InlineBackend.figure_formats = {"png", "svg"}
import pandas as pd
from IPython.display import Markdown, display

from truss_research.evaluacion.presentacion import notebook_context

context = notebook_context()
CONFIG = context.config
STUDY_DIR, REPORT_DIR = context.study_dir, context.report_dir
DATA_DIR = CONFIG.data_dir
display(
    Markdown(
        f"**Estudio:** `{STUDY_DIR.name}` — **etiqueta:** `{CONFIG.label}` — **informe:** `{REPORT_DIR.name}`"
    )
)
if CONFIG.label == "comprobacion":
    display(
        Markdown(
            "**Comprobación técnica: sus métricas no constituyen resultados de tesis.**"
        )
    )

## 1. Decisiones congeladas

`freeze.json` fija la selección, modelos, políticas y validación antes de consultar prueba. `evolution/closed.json` acredita terminación. Una familia diferente puede lograr menor error en prueba sin cambiar la elección de validación. Iniciar prueba no equivale a completar el cierre; el exportador exige cierre completo.

In [ ]:
from truss_research.datos.artefactos import read_artifact

closed = read_artifact(STUDY_DIR / "evolution" / "closed.json")
display(
    pd.Series(
        {
            "fase": closed["phase"],
            "selección_congelada": context.frozen["selected"],
            "firma_del_estudio": context.frozen["signature"],
            "prueba_usada_para_entrenar": context.frozen[
                "test_labels_used_for_training"
            ],
        }
    )
)

## 2. Resultado principal y aplicación secundaria

La familia se elige por AUC observado de validación en el menor presupuesto, con MAE como desempate. Presentar prueba no reabre esa decisión. Las comparaciones deben conservar partición, presupuesto y rol de cada control. A usa respuestas históricas; B usa estimaciones sin validación FEM.

In [ ]:
display(pd.read_csv(REPORT_DIR / "predictor_utility_summary.csv"))
display(pd.read_csv(REPORT_DIR / "predictive_model_summary.csv"))
display(pd.read_csv(REPORT_DIR / "comparison.csv"))

## 3. Evidencia y preguntas de investigación

Interpretar conjuntamente utilidad observada, error físico, dispersión entre semillas, topologías cubiertas y costo. Las OOF tienen calentamiento sin cobertura. Las semillas no son réplicas estructurales independientes; los intervalos son exploratorios y no demuestran equivalencia ni superioridad poblacional.

La evaluación temporal pertenece a una sola trayectoria ya examinada. Las categorías ausentes de prueba no tienen evaluación final en ese bloque. La interpolación por diseños únicos es una sensibilidad separada, no una validación externa nueva.

In [ ]:
display(pd.read_csv(REPORT_DIR / "category_coverage.csv"))
display(pd.read_csv(REPORT_DIR / "predictor_costs.csv"))
display(pd.read_csv(REPORT_DIR / "policy_costs.csv"))

## 4. Límites físicos y trabajo pendiente

A36, fluencia de 250 MPa, Eurocódigo 3, geometría y FEM proceden de declaraciones humanas registradas. La ficha no sustituye su evidencia original. Siguen pendientes cargas, apoyos, conectividad, catálogo y asignación de secciones, solver, dimensionamiento y agregación exacta de respuestas.

La utilización promedio no verifica cada barra; L/700 es un criterio operativo adoptado. Las configuraciones nuevas no tienen certificación estructural y no existe medición comparable del evaluador original para afirmar ahorro FEM. La estabilidad entre semillas no equivale a estabilidad cinemática.

In [ ]:
display(pd.read_csv(REPORT_DIR / "physical_provenance.csv"))
display(Markdown((REPORT_DIR / "RESULTS.md").read_text(encoding="utf-8")))

## 5. Reproducción y comunicación

Las fuentes canónicas son los cuatro archivos de `notebooks/`, conservados sin salidas. `make notebooks ESTUDIO=...` usa estas mismas fuentes y exporta notebooks ejecutados, HTML y figuras en un destino nuevo bajo `reports/`. Las firmas identifican las fuentes y las salidas de la presentación; no modifican los artefactos científicos.

Para volver a abrir las fuentes en Jupyter, definir `TRUSS_STUDY` y usar un informe vigente generado con `make report`; `TRUSS_REPORT` permite elegirlo explícitamente. `make dev ESTUDIO=...` abre la presentación exportada. Los artefactos pueden contener información restringida. Las campañas, exportaciones y suite completa se ejecutan manualmente según AGENTS.md.